# Lab 3: Conditional Probability

This notebook builds a bigram language model. A bigram is two words that sit next to each other. `bigram_counts` counts, for each word, which words come right after it. It stores them as `counts[prev_word][next_word]` (a dict inside a dict). `conditional_probabilities` turns each word's counts into chances that add up to 1: `P(next_word | prev_word)`. The reading panel explains a dict inside a dict and conditional probability in full. Here you put them to work.

Before modern LLMs, this kind of model powered autocomplete and phone keyboard suggestions. It is the same idea, except that it looks at only one word before, not a long stretch of text.

In [ ]:
%matplotlib inline
import random
import re

import matplotlib.pyplot as plt
import numpy as np

## Exercise

### Goal
Lab 1 gave one list of chances for all words. Here you build a separate list **for each previous word**: "given the word I just saw, how likely is each possible next word?". This is called **conditional probability**. It is the core idea of a language model (this one only looks back one word). You write two functions: one that counts which word follows which, and one that turns those counts into chances. `tokenize` is already written for you in the cell.

### 1. `bigram_counts(sentences)`
Code shape (replace each `...`):
```python
def bigram_counts(sentences):
    counts = {}
    for sentence in sentences:
        words = ...
        for prev, nxt in zip(..., ...):
            inner = counts.setdefault(prev, {})
            inner[nxt] = ... + 1
    return counts
```
**You get:** a list of sentences (`list[str]`), like `["a b a c"]`.
**You give back:** a dict of dicts (`dict[str, dict[str, int]]`). `counts[prev][nxt]` is how many times the word `nxt` came right after the word `prev`.
A *bigram* is two words next to each other, like `("the", "movie")`. In that pair, `the` is the previous word (`prev`) and `movie` is the next word (`nxt`). A *dict of dicts* is a dict whose values are dicts too. `counts["the"]["movie"]` first looks up `"the"`, then looks up `"movie"` inside the result.
For `["a b a c"]` the word pairs are `(a, b)`, `(b, a)`, `(a, c)`, so the result is `{"a": {"b": 1, "c": 1}, "b": {"a": 1}}`.
- Pairs never cross from one sentence into the next. The last word of one sentence is not followed by the first word of the next.
- A word that only ever ends sentences (like `c` above) gets no entry of its own, and a one-word sentence adds nothing. An empty list gives `{}`.

**Idea:** split each sentence into words with `tokenize`. Look at each pair of neighbors, and add 1 to `counts[prev][nxt]`. Make the inner dict the first time you see `prev`. `zip(words, words[1:])` gives the pairs: for `["a", "b", "c"]` it gives `("a", "b")` and `("b", "c")`. `d.setdefault(key, {})` gives back `d[key]`, and makes it an empty `{}` first if it was missing.
**Check:** `["a b", "b c"]` gives `{"a": {"b": 1}, "b": {"c": 1}}`.

### 2. `conditional_probabilities(counts)`
Code shape:
```python
def conditional_probabilities(counts):
    result = {}
    for prev, inner in counts.items():
        total = ...
        result[prev] = {nxt: ... for nxt, count in inner.items()}
    return result
```
**You get:** the dict from step 1, like `{"a": {"b": 1, "c": 1}, "b": {"a": 1}}`.
**You give back:** a dict of dicts (`dict[str, dict[str, float]]`) with the same keys. Each count is divided by the total of **its own inner dict**: `{"a": {"b": 0.5, "c": 0.5}, "b": {"a": 1.0}}`. Every inner dict adds up to 1.0.
*Conditional probability* `P(next | prev)` is the chance of the next word, given that `prev` was just seen. As a fraction: (times `next` came after `prev`) / (times anything came after `prev`).
**Idea:** for each `prev`, add up the counts in its inner dict. Then divide each count by that total. Each `prev` has its own total, not one shared total.
**Check:** `a` was followed by `b` once and by `c` once, so the total is 2, and each gets `1/2 = 0.5`. `b` was followed only by `a`, so the total is 1 and `a` gets `1.0`.

### What the asserts below test
The cell uses `"the movie was great"`, `"the movie was long"`, `"the food was great"`. Count the pairs: `the` is followed by `movie` twice and by `food` once. `was` is followed by `great` twice and by `long` once. The asserts check exactly those four counts.

Then they check that the chances after `was` add up to 1.0. They also check that `P(great | was) = 2/3`: `was` is followed by a word 3 times, and 2 of those times the word is `great`.

If everything passes, you see `All checks passed.`. If an assert fails, it shows the value it expected and the value it got.

In [ ]:
def tokenize(sentence: str) -> list[str]:
    return re.findall(r"[a-z]+", sentence.lower())


def bigram_counts(sentences: list[str]) -> dict[str, dict[str, int]]:
    """Count which word comes right after which: counts[prev_word][next_word].

    You get: a list of sentences, like ["a b a c"]
    You give back: a dict of dicts. The outer key is a word. Its inner dict says
    which words came right after it, and how many times, like
    {"a": {"b": 1, "c": 1}, "b": {"a": 1}}.
    Pairs never cross from one sentence into the next. A word that only ever
    ends sentences, or a one-word sentence, adds no outer key.

    Example:
        bigram_counts(["a b a c"]) -> {"a": {"b": 1, "c": 1}, "b": {"a": 1}}
    """
    counts: dict[str, dict[str, int]] = {}
    # TODO:
    # 1. Go through the sentences. Turn each one into a list of words
    #    with tokenize().
    # 2. Go through each pair of neighbors (prev, nxt) in that list
    #    (zip(words, words[1:]) gives you the pairs).
    # 3. Make sure counts[prev] exists (counts.setdefault(prev, {}) does
    #    that), then add 1 to its entry for nxt (start from 0).
    # 4. When all the sentences are done, return `counts`.
    raise NotImplementedError("bigram_counts() is not implemented yet - replace the TODO with your code")


def conditional_probabilities(counts: dict[str, dict[str, int]]) -> dict[str, dict[str, float]]:
    """Divide each previous word's inner dict by its own total, so its values add up to 1.

    You get: the dict from bigram_counts, like {"a": {"b": 1, "c": 1}}
    You give back: a dict of dicts with the same keys. Each count is divided by
    the total of its own inner dict, like {"a": {"b": 0.5, "c": 0.5}}.
    Each inner dict adds up to 1.0.

    Example:
        conditional_probabilities({"a": {"b": 1, "c": 1}, "b": {"a": 1}})
        -> {"a": {"b": 0.5, "c": 0.5}, "b": {"a": 1.0}}
    """
    # TODO:
    # 1. Go through each previous word and its inner dict of next-word counts.
    # 2. Add up the counts in that inner dict to get its own total.
    # 3. Make a new inner dict: each next word's count divided by that total.
    # 4. Put the new inner dicts in a result dict and return it.
    raise NotImplementedError("conditional_probabilities() is not implemented yet - replace the TODO with your code")


sample = ["the movie was great", "the movie was long", "the food was great"]
counts = bigram_counts(sample)

assert counts["the"]["movie"] == 2, 'counts["the"]["movie"] should be ' + repr(2) + ", got " + repr(counts["the"]["movie"])
assert counts["the"]["food"] == 1, 'counts["the"]["food"] should be ' + repr(1) + ", got " + repr(counts["the"]["food"])
assert counts["was"]["great"] == 2, 'counts["was"]["great"] should be ' + repr(2) + ", got " + repr(counts["was"]["great"])
assert counts["was"]["long"] == 1, 'counts["was"]["long"] should be ' + repr(1) + ", got " + repr(counts["was"]["long"])

cond = conditional_probabilities(counts)
assert abs(sum(cond["was"].values()) - 1.0) < 1e-9, 'sum(cond["was"].values()) should be about ' + repr(1.0) + ", got " + repr(sum(cond["was"].values()))
assert abs(cond["was"]["great"] - (2 / 3)) < 1e-9, 'cond["was"]["great"] should be about ' + repr(2 / 3) + ", got " + repr(cond["was"]["great"])

print("All checks passed.")

## Solution

In [ ]:
def bigram_counts(sentences: list[str]) -> dict[str, dict[str, int]]:
    counts: dict[str, dict[str, int]] = {}
    for sentence in sentences:
        words = tokenize(sentence)
        for prev, nxt in zip(words, words[1:]):
            inner = counts.setdefault(prev, {})
            inner[nxt] = inner.get(nxt, 0) + 1
    return counts


def conditional_probabilities(counts: dict[str, dict[str, int]]) -> dict[str, dict[str, float]]:
    cond: dict[str, dict[str, float]] = {}
    for prev, next_counts in counts.items():
        total = sum(next_counts.values())
        cond[prev] = {nxt: count / total for nxt, count in next_counts.items()}
    return cond


print(conditional_probabilities(bigram_counts(sample))["was"])

## Real-world project

The cells below build a bigram model from the real review sentences. They use `generate_sentence` to make new sentences one word at a time (they come out a bit odd, a little like an LLM), and draw a heatmap of `P(next | previous)` for the most common words. It is the same idea as lab 1's bar chart, but for chances that depend on the word before.

In [ ]:
"""Load the shared real-world review dataset (embedded so this notebook needs no repo checkout)."""
import csv
import io

REVIEW_CSV = """sentence,label
the movie was absolutely wonderful and kept me hooked the whole time,1
i loved every minute of this film,1
the acting was superb and the story was touching,1
this was one of the best movies i have seen all year,1
the plot was engaging and the pacing was perfect,1
a brilliant film with outstanding performances,1
the soundtrack was beautiful and matched the mood perfectly,1
i would watch this movie again without hesitation,1
the director did an amazing job telling this story,1
this film exceeded all of my expectations,1
the characters felt real and the dialogue was sharp,1
a truly moving and unforgettable experience,1
the visuals were stunning and the effects were impressive,1
this is a masterpiece that everyone should see,1
the humor was clever and the timing was excellent,1
i was on the edge of my seat the entire film,1
the ending was satisfying and well earned,1
this movie made me laugh and cry at the same time,1
a wonderfully crafted story with great performances,1
the chemistry between the leads was fantastic,1
i really enjoyed the twist near the end,1
this film is a delightful surprise from start to finish,1
the cinematography was gorgeous throughout the movie,1
a heartwarming tale that left me smiling,1
the pacing kept the tension high without ever dragging,1
i cannot recommend this movie enough,1
the product works exactly as advertised and arrived quickly,1
i am very happy with this purchase,1
the quality is excellent for the price,1
this item exceeded my expectations in every way,1
the customer service team was friendly and helpful,1
setup was quick and the instructions were clear,1
this gadget has made my daily routine so much easier,1
the build quality feels sturdy and well made,1
i would definitely buy this product again,1
the battery life is impressive and lasts all day,1
shipping was fast and the packaging was excellent,1
this is by far the best purchase i made this year,1
the design is sleek and it works flawlessly,1
i appreciate how easy this product is to use,1
the price is fair given the quality you get,1
this laptop is fast reliable and great for work,1
the headphones sound amazing and are comfortable to wear,1
i love how lightweight and portable this device is,1
the app is intuitive and runs smoothly on my phone,1
this coffee maker brews a delicious cup every morning,1
the movie was boring and dragged on for way too long,0
i did not enjoy this film at all,0
the acting felt wooden and the story made no sense,0
this was one of the worst movies i have seen all year,0
the plot was confusing and the pacing was terrible,0
a disappointing film with weak performances,0
the soundtrack was forgettable and did not fit the mood,0
i would never watch this movie again,0
the director seemed to have no clear vision for this story,0
this film fell far short of my expectations,0
the characters felt flat and the dialogue was awkward,0
a frustrating and forgettable experience overall,0
the visuals were dull and the effects looked cheap,0
this movie is a mess that nobody should waste time on,0
the humor fell flat and the timing was off,0
i kept checking the clock throughout the entire film,0
the ending felt rushed and unearned,0
this movie made me want to leave the theater early,0
a poorly crafted story with unconvincing performances,0
the chemistry between the leads felt forced,0
i was annoyed by the predictable twist near the end,0
this film is a tedious slog from start to finish,0
the cinematography was flat and uninspired throughout,0
a dull tale that left me checking my phone,0
the pacing dragged and the tension never built,0
i cannot recommend this movie to anyone,0
the product broke after only two days of use,0
i am very unhappy with this purchase,0
the quality is poor for the price charged,0
this item fell short of my expectations in every way,0
the customer service team was rude and unhelpful,0
setup was confusing and the instructions were unclear,0
this gadget has made my daily routine more frustrating,0
the build quality feels flimsy and cheaply made,0
i would never buy this product again,0
the battery life is disappointing and drains quickly,0
shipping was slow and the packaging was damaged,0
this is by far the worst purchase i made this year,0
the design is clunky and it barely works,0
i am frustrated by how hard this product is to use,0
the price is too high given the poor quality,0
this laptop is slow unreliable and bad for work,0
the headphones sound tinny and are uncomfortable to wear,0
i hate how heavy and bulky this device is,0
the app is confusing and crashes on my phone,0
this coffee maker leaks and ruins my morning,0
the sequel was even better than the original film,1
i found the story dull and the characters uninteresting,0
the special effects were groundbreaking and beautifully done,1
the remote control stopped working within a week,0
this phone case fits perfectly and looks great,1
the instructions were missing several important steps,0
the keyboard feels responsive and is a joy to type on,1
the screen arrived scratched and the seller ignored my complaint,0
this backpack is durable and has plenty of storage,1
the movie relied on tired clichés and predictable jokes,0
the restaurant served delicious food with generous portions,1
our waiter was attentive and the service was outstanding,1
the pasta was cooked perfectly and full of flavor,1
this cafe has a cozy atmosphere and friendly staff,1
the steak was tender and cooked exactly how i asked,1
dessert was the highlight of an already fantastic meal,1
the wine pairing recommendations were spot on,1
i loved the fresh ingredients and creative menu,1
the bakery bread is soft and always baked fresh,1
this bistro has become my favorite spot in town,1
the book kept me turning pages late into the night,1
the author writes with wit and genuine emotional depth,1
i finished this novel in a single weekend it was that good,1
the world building in this book is rich and immersive,1
this memoir was honest funny and deeply moving,1
the mystery had twists i never saw coming,1
the poetry collection was beautiful and thought provoking,1
i recommend this book to anyone who loves a good story,1
the hotel room was spotless and beautifully decorated,1
the staff greeted us warmly and upgraded our room,1
the pool area was relaxing and well maintained,1
check in was smooth and the concierge was helpful,1
the breakfast buffet had a wonderful variety of fresh food,1
our stay at this resort exceeded every expectation,1
the view from the balcony was absolutely breathtaking,1
the gym and spa facilities were top notch,1
this hotel is perfect for a relaxing weekend getaway,1
the car handles smoothly and the ride is comfortable,1
fuel efficiency on this vehicle has been impressive,1
the dealership made the buying process quick and painless,1
this vacuum cleaner picks up dirt with ease,1
the blender is powerful and easy to clean,1
these shoes are comfortable and have held up well,1
the jacket is warm stylish and fits perfectly,1
the restaurant food was bland and arrived cold,0
our waiter seemed annoyed and service was painfully slow,0
the pasta was overcooked and lacked any real flavor,0
this cafe was noisy and the staff were dismissive,0
the steak was tough and clearly overdone,0
dessert was stale and ruined an otherwise average meal,0
the wine pairing recommendations were completely off,0
i disliked the stale ingredients and boring menu,0
the bakery bread is dry and often stale,0
this bistro has become a place i actively avoid,0
the book dragged and i never finished reading it,0
the author writes with clumsy prose and shallow characters,0
i abandoned this novel halfway through out of boredom,0
the world building in this book felt lazy and inconsistent,0
this memoir was dull repetitive and emotionally flat,0
the mystery was predictable from the very first page,0
the poetry collection felt uninspired and forgettable,0
i would not recommend this book to anyone,0
the hotel room was dirty and poorly maintained,0
the staff ignored us and refused to help with our request,0
the pool area was filthy and badly maintained,0
check in took forever and the concierge was unhelpful,0
the breakfast buffet had stale food and few options,0
our stay at this resort fell far below expectations,0
the view from the balcony was blocked by construction,0
the gym and spa facilities were outdated and dirty,0
this hotel is a poor choice for any kind of getaway,0
the car rattles constantly and the ride is uncomfortable,0
fuel efficiency on this vehicle has been disappointing,0
the dealership made the buying process stressful and confusing,0
this vacuum cleaner barely picks up any dirt,0
the blender is weak and difficult to clean,0
these shoes are uncomfortable and fell apart quickly,0
the jacket is thin flimsy and does not fit well,0
"""

def load_reviews():
    reader = csv.DictReader(io.StringIO(REVIEW_CSV))
    sentences, labels = [], []
    for row in reader:
        sentences.append(row["sentence"])
        labels.append(int(row["label"]))
    return sentences, labels

sentences, labels = load_reviews()
print(f"Loaded {len(sentences)} real review sentences.")


In [ ]:
def generate_sentence(cond: dict[str, dict[str, float]], start: str, length: int = 10) -> str:
    words = [start]
    current = start
    for _ in range(length - 1):
        options = cond.get(current)
        if not options:
            break
        next_words = list(options.keys())
        weights = list(options.values())
        current = random.choices(next_words, weights=weights, k=1)[0]
        words.append(current)
    return " ".join(words)


counts = bigram_counts(sentences)
cond = conditional_probabilities(counts)

print("Generated sentences from the real bigram model:")
for seed in ("the", "this", "i"):
    print(f"  [{seed}] -> {generate_sentence(cond, seed)}")

total_next_counts = {prev: sum(inner.values()) for prev, inner in counts.items()}
top_prev_words = sorted(total_next_counts, key=total_next_counts.get, reverse=True)[:12]

next_word_totals: dict[str, int] = {}
for prev in top_prev_words:
    for nxt, c in counts[prev].items():
        next_word_totals[nxt] = next_word_totals.get(nxt, 0) + c
top_next_words = sorted(next_word_totals, key=next_word_totals.get, reverse=True)[:12]

matrix = np.zeros((len(top_prev_words), len(top_next_words)))
for i, prev in enumerate(top_prev_words):
    for j, nxt in enumerate(top_next_words):
        matrix[i, j] = cond[prev].get(nxt, 0.0)

plt.figure(figsize=(8, 7))
plt.imshow(matrix, cmap="viridis", aspect="auto")
plt.colorbar(label="P(next word | previous word)")
plt.xticks(range(len(top_next_words)), top_next_words, rotation=45, ha="right")
plt.yticks(range(len(top_prev_words)), top_prev_words)
plt.xlabel("next word")
plt.ylabel("previous word")
plt.title("Conditional probability heatmap (real review corpus)")
plt.tight_layout()
plt.show()